# 02 · Classical text classification baseline

The objective is a reproducible reference for English and German `human`, `ai`, and `fusion` detection. The experiment compares word TF-IDF, character TF-IDF and their sparse concatenation using logistic regression and a linear SVM. Only `text` is predictive input; language and domain are used for stratification and diagnostics.

Research questions: do character n-grams improve CV macro-F1, does combining representations help, and how stable are results across three fixed folds? No method is presumed superior before measurement.

Completed Kaggle measurements are restored below without fitting a model. The saved run is identified by its source data, frozen split, configuration and per-fold receipts.

## 01 · Execution stages and visible configuration

The default run loads training data, creates or restores the frozen ID manifest, and displays its audit. The three training flags below are independent explicit stages. CV uses a single frozen subset of approximately 30k development records to bound CPU cost; each selected holdout model is refitted on all development records. Vocabulary caps bound final sparse feature width, but temporary vocabulary construction and full-development fitting can still require substantial RAM.

With all training flags false, result sections load compatible saved measurements when present and otherwise remain explicitly unexecuted. No model weights are downloaded and no test file is read in that mode.

`SAVED_KAGGLE_ROOT` selects read-only review of downloaded Kaggle outputs. Set it to `None` for the original local execution stages. Training switches must remain false while reviewing a hosted run.

In [1]:
import hashlib
import json
import sys
import platform
from dataclasses import asdict
from importlib.metadata import version
from pathlib import Path

import plotly.express as px
import plotly.io
import polars as pl
from IPython.display import display
from tqdm.notebook import tqdm

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from hm3.baseline import (
    BaselineCandidate,
    fit_holdout,
    fit_or_restore_final_pipeline,
    run_cross_validation,
    summarize_cv,
)
from hm3.evaluation import classification_metrics, export_submission, subgroup_metrics
from hm3.validation import LABELS, fingerprint_file, get_or_create_manifest, load_training_data

SAVED_KAGGLE_ROOT: Path | None = ROOT / "runs/kaggle/notebook53491f866f/hm3_baseline"
plotly.io.renderers.default = "plotly_mimetype"
SEED = 2026
TRAIN_PATH = ROOT / "data/train.csv"
TEST_PATH = ROOT / "data/test.csv"
SAMPLE_PATH = ROOT / "data/sample_submission.csv"
MANIFEST_PATH = ROOT / "data/metadata/validation_manifest.parquet"
CV_SAMPLE_TARGET = 30000
FEATURE_OPTIONS = {"word_max_features": 40000, "char_max_features": 60000, "min_df": 3}
RUN_CV = False
RUN_HOLDOUT = False
RUN_SUBMISSION = False
ENVIRONMENT = {
    "python": platform.python_version(),
    **{name: version(name) for name in ["scikit-learn", "scipy", "numpy", "polars", "joblib"]},
}
ENVIRONMENT

{'python': '3.14.8',
 'scikit-learn': '1.9.1',
 'scipy': '1.18.1',
 'numpy': '2.5.5',
 'polars': '2.1.0',
 'joblib': '1.6.0'}

## 02 · Frozen holdout and CV design

Known duplicate groups are SHA-256 hashes of NFKC-normalized text with collapsed whitespace. This normalization is only for grouping; vectorizers receive the original texts. The holdout is the first of five shuffled stratified group folds, targeting 20%. The remaining 80% has three separate stratified group folds. Sorting by ID before allocation makes the split independent of CSV row order.

The target stratum is label × language × domain. If independent groups are too rare, the recorded fallback is label × language, then label; insufficient class groups fail clearly. The fixed CV sample is drawn by group within each assigned fold and joint stratum. Its target size is approximate because groups are indivisible and every observed bucket is represented. Related rewrites can still cross boundaries because reliable source-family identifiers are unavailable.

Existing assignments are never silently regenerated: source hash, IDs, labels, group hashes, manifest fingerprint and recipe must match. Keep both manifest files together when restoring local metadata.

In [2]:
training_data = load_training_data(TRAIN_PATH)
source_sha256 = fingerprint_file(TRAIN_PATH)
manifest, split_metadata = get_or_create_manifest(
    training_data, MANIFEST_PATH, source_sha256, seed=SEED, cv_sample_size=CV_SAMPLE_TARGET
)
audit = json.loads(MANIFEST_PATH.with_suffix(".json").read_text())["audit"]
display(
    {
        key: split_metadata[key]
        for key in [
            "source_sha256",
            "manifest_sha256",
            "holdout_method",
            "holdout_stratification",
            "cv_method",
            "cv_stratification",
            "cv_sample_method",
        ]
    }
)
display(
    pl.DataFrame(audit["partition_counts"]),
    pl.DataFrame(audit["fold_counts"]),
    pl.DataFrame(audit["cv_sample_fold_counts"]),
)

{'source_sha256': 'd55823815042eed2937e92e1018f5595915b8f7573d861048956521434f65025',
 'manifest_sha256': '1f12f2a2b312b513ace36966e9b3a011e7e3b279aeed6dd254173ac9a3d26a5d',
 'holdout_method': 'first of five shuffled StratifiedGroupKFold folds',
 'holdout_stratification': ['label', 'lang', 'domain'],
 'cv_method': 'three shuffled StratifiedGroupKFold folds within development',
 'cv_stratification': ['label', 'lang', 'domain'],
 'cv_sample_method': 'seeded group sampling within CV fold and joint strata; approximate target'}

partition,len
str,i64
"""development""",217881
"""validation""",54471


cv_fold,len
i64,i64
0,72628
1,72627
2,72626


cv_fold,len
i64,i64
0,10001
1,9999
2,10002


In [3]:
for grouping in ["class", "language", "domain", "joint"]:
    display(pl.DataFrame(audit[f"{grouping}_counts"]))
display(pl.DataFrame(audit["distribution_deviations"]))
display(
    {
        key: audit[key]
        for key in ["duplicate_groups", "known_group_partition_overlap", "known_group_fold_overlap"]
    }
)
fold_joint_counts = pl.DataFrame(audit["cv_joint_counts"])
sample_joint_counts = pl.DataFrame(audit["cv_sample_joint_counts"])
display(
    fold_joint_counts.pivot(on="cv_fold", index=["label", "lang", "domain"], values="len").sort(
        ["lang", "domain", "label"]
    )
)
display(
    sample_joint_counts.pivot(on="cv_fold", index=["label", "lang", "domain"], values="len").sort(
        ["lang", "domain", "label"]
    )
)
composition = (
    manifest.group_by(["partition", "label"])
    .len()
    .with_columns((100 * pl.col("len") / pl.col("len").sum().over("partition")).alias("percent"))
)
display(
    px.bar(
        composition,
        x="partition",
        y="percent",
        color="label",
        barmode="stack",
        title="Class composition of the frozen holdout split",
        labels={"partition": "Partition", "percent": "Within-partition percent", "label": "Class"},
    )
)

partition,label,len
str,str,i64
"""development""","""ai""",97554
"""development""","""fusion""",66944
"""development""","""human""",53383
"""validation""","""ai""",24386
"""validation""","""fusion""",16738
"""validation""","""human""",13347


partition,lang,len
str,str,i64
"""development""","""de-DE""",42217
"""development""","""en-EN""",175664
"""validation""","""de-DE""",10554
"""validation""","""en-EN""",43917


partition,domain,len
str,str,i64
"""development""","""arxiv_papers""",11431
"""development""","""blog_authorship_corpus""",32437
"""development""","""bundestag""",41874
"""development""","""cnn_news""",35428
"""development""","""euro_court_cases""",22062
…,…,…
"""validation""","""cnn_news""",8857
"""validation""","""euro_court_cases""",5516
"""validation""","""gutenberg""",1468


partition,label,lang,domain,len
str,str,str,str,i64
"""development""","""ai""","""de-DE""","""bundestag""",15982
"""development""","""ai""","""de-DE""","""gutenberg""",288
"""development""","""ai""","""en-EN""","""arxiv_papers""",9640
"""development""","""ai""","""en-EN""","""blog_authorship_corpus""",12547
"""development""","""ai""","""en-EN""","""cnn_news""",15590
…,…,…,…,…
"""validation""","""human""","""en-EN""","""cnn_news""",2679
"""validation""","""human""","""en-EN""","""euro_court_cases""",1256
"""validation""","""human""","""en-EN""","""gutenberg""",43


scope,grouping,max_deviation_pp
str,str,f64
"""holdout""","""class""",0.004165
"""cv""","""class""",0.003334
"""cv_sample""","""class""",0.004083
"""cv_sample_vs_development""","""class""",0.003033
"""holdout""","""language""",0.000581
…,…,…
"""cv_sample_vs_development""","""domain""",0.006385
"""holdout""","""joint""",0.001494
"""cv""","""joint""",0.001019


{'duplicate_groups': 1,
 'known_group_partition_overlap': 0,
 'known_group_fold_overlap': 0}

label,lang,domain,0,1,2
str,str,str,i64,i64,i64
"""ai""","""de-DE""","""bundestag""",5327,5327,5328
"""fusion""","""de-DE""","""bundestag""",4770,4770,4769
"""human""","""de-DE""","""bundestag""",3861,3861,3861
"""ai""","""de-DE""","""gutenberg""",96,96,96
"""fusion""","""de-DE""","""gutenberg""",10,10,10
…,…,…,…,…,…
"""fusion""","""en-EN""","""house_of_commons""",2562,2562,2562
"""human""","""en-EN""","""house_of_commons""",1854,1854,1854
"""ai""","""en-EN""","""student_essays""",5756,5755,5756


label,lang,domain,0,1,2
str,str,str,i64,i64,i64
"""ai""","""de-DE""","""bundestag""",733,733,734
"""fusion""","""de-DE""","""bundestag""",657,657,657
"""human""","""de-DE""","""bundestag""",532,532,532
"""ai""","""de-DE""","""gutenberg""",13,13,13
"""fusion""","""de-DE""","""gutenberg""",1,1,1
…,…,…,…,…,…
"""fusion""","""en-EN""","""house_of_commons""",353,353,353
"""human""","""en-EN""","""house_of_commons""",255,255,255
"""ai""","""en-EN""","""student_essays""",793,792,793


Maximum deviations compare each holdout partition to all training records, each CV fold to development, and each sampled CV fold to the sampled population. A separate comparison measures sample composition against all development records. Units are percentage points. No known normalized group may overlap partitions or CV folds. These checks do not establish independence between different rewrites of the same underlying source.

## 03 · Representations and controlled experiment matrix

Word features use unigrams and bigrams. Character features use character 3–5-grams, including punctuation and whitespace. The scikit-learn character analyzer collapses runs of consecutive whitespace before constructing n-grams; it does not preserve exact whitespace runs. Both use sublinear term frequency, minimum document frequency 3, float32 storage and L2 normalization. Case is preserved; no stopwords, stemming, accent removal or duplicate-normalization transform is applied to model input. The combined representation concatenates the two separately normalized sparse blocks with equal weights; it is not converted to a dense matrix.

The four candidates compare representation and classifier with C fixed at 1.0. Additional regularization settings are deferred until the initial comparison is measured. Logistic regression uses `saga`, tolerance 0.001, at most 300 iterations and no class weights. Linear SVM uses `dual="auto"`, tolerance 0.001, at most 5,000 iterations and no class weights. Both use seed 2026. Class weights can be tested later as a separate controlled experiment.

In [4]:
CANDIDATES = [
    BaselineCandidate("word_tfidf_logistic_regression_c1", "word", "logistic_regression", 1.0),
    BaselineCandidate("char_tfidf_logistic_regression_c1", "char", "logistic_regression", 1.0),
    BaselineCandidate(
        "word_char_tfidf_logistic_regression_c1", "word_char", "logistic_regression", 1.0
    ),
    BaselineCandidate("word_char_tfidf_linear_svm_c1", "word_char", "linear_svm", 1.0),
]
display(pl.DataFrame([asdict(candidate) for candidate in CANDIDATES]))
recipe = {
    "seed": SEED,
    "source_sha256": source_sha256,
    "manifest_sha256": split_metadata["manifest_sha256"],
    "feature_options": FEATURE_OPTIONS,
    "candidates": [asdict(candidate) for candidate in CANDIDATES],
    "environment": ENVIRONMENT,
    "source_modules": {
        name: fingerprint_file(ROOT / f"src/hm3/{name}.py")
        for name in ["baseline", "evaluation", "validation"]
    },
}
recipe_sha256 = hashlib.sha256(json.dumps(recipe, sort_keys=True).encode()).hexdigest()
RESULTS_PATH = ROOT / "runs/classical_baseline" / recipe_sha256[:12]
CV_RECORD = RESULTS_PATH / "cv_record.json"
HOLDOUT_RECORD = RESULTS_PATH / "holdout_record.json"
if SAVED_KAGGLE_ROOT is not None:
    if RUN_CV or RUN_HOLDOUT or RUN_SUBMISSION:
        raise ValueError(
            "Saved Kaggle review is read-only; disable training and submission switches"
        )
    pointer = json.loads((SAVED_KAGGLE_ROOT / "current_run.json").read_text())
    RESULTS_PATH = SAVED_KAGGLE_ROOT / pointer["run_directory"]
    saved_run = json.loads((RESULTS_PATH / "run_metadata.json").read_text())
    saved_recipe = saved_run["recipe"]
    expected_settings = {
        "seed": SEED,
        "source_sha256": source_sha256,
        "manifest_sha256": split_metadata["manifest_sha256"],
        "features": FEATURE_OPTIONS,
        "candidates": [asdict(candidate) for candidate in CANDIDATES],
    }
    if any(saved_recipe[key] != value for key, value in expected_settings.items()):
        raise ValueError("Downloaded baseline uses a different dataset, split or experiment matrix")
    recipe, recipe_sha256 = saved_recipe, saved_run["recipe_sha256"]
    CV_RECORD = RESULTS_PATH / "cv_record.json"
    HOLDOUT_RECORD = RESULTS_PATH / "holdout_record.json"
    display({"measured_environment": saved_recipe["environment"]})
display({"recipe_sha256": recipe_sha256, "results": str(RESULTS_PATH.relative_to(ROOT))})

name,representation,classifier,c
str,str,str,f64
"""word_tfidf_logistic_regression…","""word""","""logistic_regression""",1.0
"""char_tfidf_logistic_regression…","""char""","""logistic_regression""",1.0
"""word_char_tfidf_logistic_regre…","""word_char""","""logistic_regression""",1.0
"""word_char_tfidf_linear_svm_c1""","""word_char""","""linear_svm""",1.0


{'measured_environment': {'python': '3.13.18',
  'scikit-learn': '1.9.1',
  'numpy': '2.5.5',
  'scipy': '1.18.1',
  'polars': '2.1.0',
  'joblib': '1.6.0'}}

{'recipe_sha256': 'd36d5b40b9bdbc3875a8cf8c5287982736b222f710b00f21cfb24bdac28674ee',
 'results': 'runs/kaggle/notebook53491f866f/hm3_baseline/runs/classical_baseline/d36d5b40b9bdbc3875a8cf8c5287982736b222f710b00f21cfb24bdac28674ee'}

## 04 · Three-fold CV and model comparison

Each fold fits a fresh pipeline on the other two folds of the same frozen CV sample. CV validation rows and the fixed holdout never participate in vocabulary or IDF fitting. Candidates run sequentially to limit concurrent sparse matrices.

Selection uses mean fold three-class macro-F1, with lexical experiment-name ordering to break an exact tie. Sample standard deviation uses $n-1$ degrees of freedom. Pooled OOF macro-F1 is reported separately. Coefficient memory is a lower bound for model storage, not peak process RAM. Iteration-limit flags require investigation before interpreting the selected model as converged.

In [5]:
cv_scores: pl.DataFrame | None = None
cv_predictions: pl.DataFrame | None = None
cv_comparison: pl.DataFrame | None = None
if SAVED_KAGGLE_ROOT is not None:
    cv_scores = pl.read_parquet(RESULTS_PATH / "cv_scores.parquet")
    cv_predictions = pl.read_parquet(RESULTS_PATH / "cv_predictions.parquet")
    expected_cv = (
        manifest.filter(pl.col("is_cv_sample"))
        .select("id", "label", pl.col("cv_fold").alias("fold"))
        .sort("id")
    )
    for candidate in CANDIDATES:
        population = cv_predictions.filter(pl.col("candidate") == candidate.name).sort("id")
        if not population.select("id", "label", "fold").equals(expected_cv):
            raise ValueError("Downloaded OOF IDs, labels or folds differ from the frozen sample")
        for fold in range(3):
            directory = RESULTS_PATH / "cv_folds" / candidate.name
            receipt = json.loads((directory / f"fold_{fold}.json").read_text())
            if (
                receipt["identity"]["run_identity"] != recipe_sha256
                or receipt["identity"]["candidate"] != asdict(candidate)
                or fingerprint_file(directory / f"fold_{fold}.parquet")
                != receipt["predictions_sha256"]
            ):
                raise ValueError("Downloaded CV fold identity or checksum mismatch")
            score = cv_scores.filter(
                (pl.col("candidate") == candidate.name) & (pl.col("fold") == fold)
            ).row(0, named=True)
            fold_predictions = population.filter(pl.col("fold") == fold)
            measured = classification_metrics(
                fold_predictions["label"].to_list(), fold_predictions["prediction"].to_list()
            )["macro_f1"]
            if score != receipt["score"] or measured != score["macro_f1"]:
                raise ValueError("Downloaded CV score does not match its OOF predictions")
elif CV_RECORD.exists():
    record = json.loads(CV_RECORD.read_text())
    if record["recipe_sha256"] != recipe_sha256:
        raise ValueError("Saved CV recipe is incompatible")
    for name, expected_hash in record["files"].items():
        if fingerprint_file(RESULTS_PATH / name) != expected_hash:
            raise ValueError(f"Saved CV artifact changed: {name}")
    cv_scores = pl.read_parquet(RESULTS_PATH / "cv_scores.parquet")
    cv_predictions = pl.read_parquet(RESULTS_PATH / "cv_predictions.parquet")
elif RUN_CV:
    RESULTS_PATH.mkdir(parents=True, exist_ok=True)
    measured_scores, measured_predictions = [], []
    for candidate in tqdm(CANDIDATES, desc="Candidates, three folds each"):
        scores, predictions = run_cross_validation(
            training_data, manifest, [candidate], **FEATURE_OPTIONS
        )
        measured_scores.append(scores)
        measured_predictions.append(predictions)
    cv_scores, cv_predictions = pl.concat(measured_scores), pl.concat(measured_predictions)
    cv_scores.write_parquet(RESULTS_PATH / "cv_scores.parquet")
    cv_predictions.write_parquet(RESULTS_PATH / "cv_predictions.parquet")
    record = {
        "recipe_sha256": recipe_sha256,
        "recipe": recipe,
        "files": {
            name: fingerprint_file(RESULTS_PATH / name)
            for name in ["cv_scores.parquet", "cv_predictions.parquet"]
        },
    }
    temporary = CV_RECORD.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(record, indent=2))
    temporary.replace(CV_RECORD)
if cv_scores is not None and cv_predictions is not None:
    cv_comparison = summarize_cv(cv_scores, cv_predictions)
    with pl.Config(tbl_rows=16, tbl_cols=12):
        display(cv_scores, cv_comparison)
else:
    display(
        {
            "cv_status": "not executed",
            "candidates": len(CANDIDATES),
            "fits_required": 3 * len(CANDIDATES),
        }
    )

candidate,representation,classifier,c,fold,train_rows,…,macro_f1,fit_seconds,predict_seconds,features,coefficient_mib,hit_iteration_limit
str,str,str,f64,i64,i64,…,f64,f64,f64,i64,f64,bool
"""word_tfidf_logistic_regression…","""word""","""logistic_regression""",1.0,0,20001,…,0.828258,42.131816,8.942316,40000,0.457764,false
"""word_tfidf_logistic_regression…","""word""","""logistic_regression""",1.0,1,20003,…,0.825932,41.999186,9.188693,40000,0.457764,false
"""word_tfidf_logistic_regression…","""word""","""logistic_regression""",1.0,2,20000,…,0.82534,42.442468,9.344167,40000,0.457764,false
"""char_tfidf_logistic_regression…","""char""","""logistic_regression""",1.0,0,20001,…,0.843404,197.084448,71.214344,60000,0.686646,false
"""char_tfidf_logistic_regression…","""char""","""logistic_regression""",1.0,1,20003,…,0.839854,198.532779,72.31262,60000,0.686646,false
"""char_tfidf_logistic_regression…","""char""","""logistic_regression""",1.0,2,20000,…,0.842186,202.462815,75.106714,60000,0.686646,false
"""word_char_tfidf_logistic_regre…","""word_char""","""logistic_regression""",1.0,0,20001,…,0.850089,252.824858,82.530838,100000,1.144409,false
"""word_char_tfidf_logistic_regre…","""word_char""","""logistic_regression""",1.0,1,20003,…,0.845619,245.796008,82.552179,100000,1.144409,false
"""word_char_tfidf_logistic_regre…","""word_char""","""logistic_regression""",1.0,2,20000,…,0.848279,253.175789,83.736231,100000,1.144409,false


candidate,representation,classifier,c,cv_macro_f1_mean,cv_macro_f1_std,total_fit_seconds,max_coefficient_mib,hit_iteration_limit,pooled_oof_macro_f1
str,str,str,f64,f64,f64,f64,f64,bool,f64
"""word_char_tfidf_logistic_regre…","""word_char""","""logistic_regression""",1.0,0.847996,0.002248,751.796655,1.144409,false,0.847997
"""char_tfidf_logistic_regression…","""char""","""logistic_regression""",1.0,0.841815,0.001804,598.080041,0.686646,false,0.841818
"""word_char_tfidf_linear_svm_c1""","""word_char""","""linear_svm""",1.0,0.831013,0.00349,696.905245,2.288818,false,0.831016
"""word_tfidf_logistic_regression…","""word""","""logistic_regression""",1.0,0.82651,0.001543,126.57347,0.457764,false,0.826508


## 05 · CV-selected model and fixed-validation evaluation

Only the highest mean CV macro-F1 configuration is eligible for primary holdout evaluation. It is fitted on all development texts, including development records outside the CV sample. A completed evaluation is restored from fingerprinted outputs rather than refitted on rerun. Repeatedly changing the experiment after inspecting holdout results would turn this set into a tuning set; any such later comparison must be labeled exploratory.

In [6]:
selected: BaselineCandidate | None = None
holdout_predictions: pl.DataFrame | None = None
holdout_metrics: dict[str, object] | None = None
if cv_comparison is not None:
    best_name = cv_comparison["candidate"][0]
    selected = next(candidate for candidate in CANDIDATES if candidate.name == best_name)
    display(asdict(selected))
if RUN_HOLDOUT and selected is None:
    raise ValueError("Complete CV selection before holdout fitting")
if HOLDOUT_RECORD.exists() and selected is not None:
    record = json.loads(HOLDOUT_RECORD.read_text())
    recorded_candidate = record["candidate"]
    candidate_matches = (
        recorded_candidate == asdict(selected)
        if SAVED_KAGGLE_ROOT is not None
        else recorded_candidate == selected.name
    )
    if record["recipe_sha256"] != recipe_sha256 or not candidate_matches:
        raise ValueError("Saved holdout evaluation is incompatible")
    for name, expected_hash in record["files"].items():
        if fingerprint_file(RESULTS_PATH / name) != expected_hash:
            raise ValueError(f"Saved holdout artifact changed: {name}")
    holdout_predictions = pl.read_parquet(RESULTS_PATH / "holdout_predictions.parquet")
    holdout_metrics = json.loads((RESULTS_PATH / "holdout_metrics.json").read_text())
    expected_holdout = (
        manifest.filter(pl.col("partition") == "validation").select("id", "label").sort("id")
    )
    if not holdout_predictions.select("id", "label").sort("id").equals(expected_holdout):
        raise ValueError("Downloaded holdout IDs or labels differ from the frozen validation split")
    recalculated = classification_metrics(
        holdout_predictions["label"].to_list(), holdout_predictions["prediction"].to_list()
    )
    if any(holdout_metrics[key] != value for key, value in recalculated.items()):
        raise ValueError("Downloaded holdout metrics do not match saved predictions")
elif RUN_HOLDOUT and selected is not None:
    fitted_pipeline, holdout_predictions, holdout_metrics = fit_holdout(
        training_data, manifest, selected, **FEATURE_OPTIONS
    )
    holdout_predictions.write_parquet(RESULTS_PATH / "holdout_predictions.parquet")
    (RESULTS_PATH / "holdout_metrics.json").write_text(json.dumps(holdout_metrics, indent=2))
    record = {
        "recipe_sha256": recipe_sha256,
        "candidate": selected.name,
        "files": {
            name: fingerprint_file(RESULTS_PATH / name)
            for name in ["holdout_predictions.parquet", "holdout_metrics.json"]
        },
    }
    temporary = HOLDOUT_RECORD.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(record, indent=2))
    temporary.replace(HOLDOUT_RECORD)
if holdout_metrics is not None:
    display(
        {
            key: holdout_metrics[key]
            for key in [
                "macro_f1",
                "weighted_f1",
                "accuracy",
                "training_rows",
                "validation_rows",
                "model_class_order",
            ]
        }
    )
    display(pl.DataFrame(holdout_metrics["per_class"]))
else:
    display({"holdout_status": "not executed"})

{'name': 'word_char_tfidf_logistic_regression_c1',
 'representation': 'word_char',
 'classifier': 'logistic_regression',
 'c': 1.0}

{'macro_f1': 0.895719920476218,
 'weighted_f1': 0.912280874161554,
 'accuracy': 0.9122836004479448,
 'training_rows': 217881,
 'validation_rows': 54471,
 'model_class_order': ['ai', 'fusion', 'human']}

label,precision,recall,f1,support
str,f64,f64,f64,i64
"""human""",0.811607,0.87278,0.841083,13347
"""ai""",0.987024,0.991881,0.989446,24386
"""fusion""",0.887522,0.827817,0.856631,16738


## 06 · Confusion matrices and subgroup diagnostics

The metric class order is `human`, `ai`, `fusion`. Estimators learn string labels and their own class order is recorded explicitly. Macro-F1 always averages these three labels with `zero_division=0`, even when a subgroup lacks a class. Missing classes and groups with fewer than 100 records are flagged; the threshold is a reporting convention, not a confidence guarantee. Length buckets are whitespace-word counts, not tokenizer lengths.

In [7]:
if holdout_metrics is not None and holdout_predictions is not None:
    for key, title in [
        ("confusion_matrix", "Fixed-validation confusion matrix: counts"),
        ("row_normalized_confusion_matrix", "Fixed-validation confusion matrix: row proportions"),
    ]:
        display(
            px.imshow(
                holdout_metrics[key],
                x=list(LABELS),
                y=list(LABELS),
                text_auto=True,
                title=title,
                labels={"x": "Predicted class", "y": "True class", "color": "Value"},
            )
        )
    for columns in [["lang"], ["domain"], ["length_bucket"], ["lang", "domain"]]:
        diagnostics = subgroup_metrics(holdout_predictions, columns)
        if SAVED_KAGGLE_ROOT is None:
            diagnostics.write_json(RESULTS_PATH / f"subgroup_{'_'.join(columns)}.json")
        display(diagnostics)

lang,support,observed_classes,has_sufficient_support,macro_f1,human_support,ai_support,fusion_support
str,i64,i64,bool,f64,i64,i64,i64
"""de-DE""",10554,3,true,0.925438,2902,4067,3585
"""en-EN""",43917,3,true,0.887601,10445,20319,13153


domain,support,observed_classes,has_sufficient_support,macro_f1,human_support,ai_support,fusion_support
str,i64,i64,bool,f64,i64,i64,i64
"""arxiv_papers""",2859,3,true,0.903198,82,2410,367
"""blog_authorship_corpus""",8110,3,true,0.758818,2493,3137,2480
"""bundestag""",10469,3,true,0.925746,2896,3995,3578
"""cnn_news""",8857,3,true,0.935907,2679,3898,2280
"""euro_court_cases""",5516,3,true,0.927828,1256,2228,2032
"""gutenberg""",1468,3,true,0.786636,49,1343,76
"""house_of_commons""",6372,3,true,0.902439,1391,3059,1922
"""student_essays""",10820,3,true,0.912362,2501,4316,4003


length_bucket,support,observed_classes,has_sufficient_support,macro_f1,human_support,ai_support,fusion_support
str,i64,i64,bool,f64,i64,i64,i64
"""001-128""",5520,3,true,0.800617,1517,2461,1542
"""1025+""",4698,3,true,0.912023,1597,1107,1994
"""129-512""",25240,3,true,0.888134,5596,11488,8156
"""513-1024""",19013,3,true,0.928037,4637,9330,5046


lang,domain,support,observed_classes,has_sufficient_support,macro_f1,human_support,ai_support,fusion_support
str,str,i64,i64,bool,f64,i64,i64,i64
"""de-DE""","""bundestag""",10469,3,true,0.925746,2896,3995,3578
"""de-DE""","""gutenberg""",85,3,false,0.729183,6,72,7
"""en-EN""","""arxiv_papers""",2859,3,true,0.903198,82,2410,367
"""en-EN""","""blog_authorship_corpus""",8110,3,true,0.758818,2493,3137,2480
"""en-EN""","""cnn_news""",8857,3,true,0.935907,2679,3898,2280
"""en-EN""","""euro_court_cases""",5516,3,true,0.927828,1256,2228,2032
"""en-EN""","""gutenberg""",1383,3,true,0.789873,43,1271,69
"""en-EN""","""house_of_commons""",6372,3,true,0.902439,1391,3059,1922
"""en-EN""","""student_essays""",10820,3,true,0.912362,2501,4316,4003


## 07 · Deterministic error sample

The table below selects up to two errors per observed true/predicted class pair with seed 2026. It is illustrative rather than a prevalence estimate. Confusion counts quantify which mistakes are common; short excerpts alone cannot establish causes. No error text is used to revise the selected pipeline in this experiment.

In [8]:
if holdout_predictions is not None:
    errors = holdout_predictions.filter(pl.col("label") != pl.col("prediction"))
    error_counts = errors.group_by(["label", "prediction"]).len().sort("len", descending=True)
    display(error_counts)
    if errors.height:
        examples = (
            errors.group_by(["label", "prediction"], maintain_order=True)
            .map_groups(lambda group: group.sample(n=min(2, group.height), seed=SEED))
            .join(
                training_data.select("id", pl.col("text").str.slice(0, 180).alias("excerpt")),
                on="id",
            )
        )
        display(examples.select("id", "label", "prediction", "lang", "domain", "words", "excerpt"))

label,prediction,len
str,str,u32
"""fusion""","""human""",2631
"""human""","""fusion""",1631
"""fusion""","""ai""",251
"""ai""","""fusion""",125
"""ai""","""human""",73
"""human""","""ai""",67


id,label,prediction,lang,domain,words,excerpt
str,str,str,str,str,u32,str
"""9d69f8""","""human""","""fusion""","""de-DE""","""bundestag""",398,"""Frau Präsidentin! Meine sehr v…"
"""6b2a3c""","""fusion""","""human""","""en-EN""","""blog_authorship_corpus""",111,"""I'm guessing you'll be reading…"
"""71f249""","""ai""","""fusion""","""en-EN""","""blog_authorship_corpus""",112,"""Growing up under my grandfathe…"
"""793e15""","""human""","""ai""","""en-EN""","""student_essays""",361,"""Dear Principal I Think That It…"
"""a005c9""","""fusion""","""human""","""en-EN""","""cnn_news""",856,"""By . Matt Chorley, Mailonline …"
…,…,…,…,…,…,…
"""a489f9""","""fusion""","""ai""","""en-EN""","""cnn_news""",648,"""This deep affection for all th…"
"""79957a""","""fusion""","""ai""","""en-EN""","""blog_authorship_corpus""",72,"""Five bucks to anyone who can p…"
"""8ff1e1""","""ai""","""human""","""de-DE""","""bundestag""",67,"""„Meine Kollegen aus der CDU/CS…"


## 08 · Final comparison and interpretation

Only the CV-selected model receives a primary holdout score. Empty holdout entries for other candidates mean they were not evaluated there. Compare representation effects using the CV table, including fold variability and convergence flags, before interpreting individual differences. In particular, a character-feature benefit must be measured rather than assumed.

In [9]:
if cv_comparison is not None:
    final_comparison = cv_comparison.with_columns(
        pl.when(pl.col("candidate") == (selected.name if selected is not None else ""))
        .then(pl.lit(holdout_metrics["macro_f1"] if holdout_metrics is not None else None))
        .otherwise(None)
        .alias("holdout_macro_f1")
    )
    display(final_comparison)
else:
    display({"conclusion_status": "pending measured CV and holdout results"})

candidate,representation,classifier,c,cv_macro_f1_mean,cv_macro_f1_std,total_fit_seconds,max_coefficient_mib,hit_iteration_limit,pooled_oof_macro_f1,holdout_macro_f1
str,str,str,f64,f64,f64,f64,f64,bool,f64,f64
"""word_char_tfidf_logistic_regre…","""word_char""","""logistic_regression""",1.0,0.847996,0.002248,751.796655,1.144409,false,0.847997,0.89572
"""char_tfidf_logistic_regression…","""char""","""logistic_regression""",1.0,0.841815,0.001804,598.080041,0.686646,false,0.841818,null
"""word_char_tfidf_linear_svm_c1""","""word_char""","""linear_svm""",1.0,0.831013,0.00349,696.905245,2.288818,false,0.831016,null
"""word_tfidf_logistic_regression…","""word""","""logistic_regression""",1.0,0.82651,0.001543,126.57347,0.457764,false,0.826508,null


**Measured observations.** Word + character TF-IDF with logistic regression (`C=1`)
leads the three-fold comparison at **0.847996 ± 0.002248** macro-F1. Character-only
LR reaches 0.841815 and word-only LR 0.826510. The combination improves over word-only
by 0.021486 and over character-only by 0.006181 in this shared CV population.
The combined linear SVM reaches 0.831013. No fold reports an iteration-limit hit.
These are observed differences across three fixed folds, not significance claims.

After fitting the CV-selected LR on all **217,881 development texts**, fixed-holdout
macro-F1 is **0.895720**, accuracy 0.912284 and weighted-F1 0.912281. Human has the
lowest class F1 (0.841083), followed by fusion (0.856631); AI F1 is 0.989446. Human/fusion
confusion accounts for 4,262 of 4,778 mistakes. CV and holdout use different training
sizes, so their score gap does not by itself establish leakage or distribution shift.

German macro-F1 is 0.925438 and English 0.887601. These languages have different domain
composition, so this gap is not an isolated language effect. Blogs are the weakest
aggregate domain (0.758818); Gutenberg reaches 0.786636 with only 49 human and 76 fusion
holdout documents. Unknown source-family leakage and the smaller CV population remain
limitations. Future parameter selection must continue to use development data only.

## 09 · Optional all-data refit and submission export

This separate stage requires completed CV selection and fixed-validation evaluation. It refits the selected recipe on all labeled training data and predicts test texts in bounded batches. Test data is never used to fit vocabulary, IDF or classifier parameters. The exported CSV has exactly 25,000 unique test IDs, `id,label` columns, lowercase labels and sample-submission order. Model, CSV, input and recipe fingerprints accompany the export. There is no automatic Kaggle submission.

Provide the competition test and sample files at the configured paths before enabling this stage. Reuse of a completed export requires matching input, recipe, model and CSV hashes; a verified final checkpoint can restart inference after interruption without refitting. Orphaned checkpoints with missing provenance fail clearly.

In [10]:
if RUN_SUBMISSION:
    if selected is None or holdout_metrics is None:
        raise ValueError("Complete CV selection and fixed-validation evaluation first")
    test_data = pl.read_csv(TEST_PATH, schema_overrides={"id": pl.String})
    sample_submission = pl.read_csv(SAMPLE_PATH, schema_overrides={"id": pl.String})
    submission_path = RESULTS_PATH / "submissions" / f"{selected.name}_all_train.csv"
    model_path = RESULTS_PATH / "submissions" / f"{selected.name}_all_train.joblib"
    input_identity = {
        "source_sha256": source_sha256,
        "test_sha256": fingerprint_file(TEST_PATH),
        "sample_sha256": fingerprint_file(SAMPLE_PATH),
        "recipe_sha256": recipe_sha256,
    }
    if submission_path.exists() and submission_path.with_suffix(".json").exists():
        saved = json.loads(submission_path.with_suffix(".json").read_text())
        if (
            any(saved[key] != value for key, value in input_identity.items())
            or saved["model_sha256"] != fingerprint_file(model_path)
            or saved["csv_sha256"] != fingerprint_file(submission_path)
        ):
            raise ValueError("Completed submission has incompatible inputs or outputs")
        display({"submission_status": "restored verified export", "rows": saved["rows"]})
    else:
        final_pipeline, model_record = fit_or_restore_final_pipeline(
            training_data, selected, model_path, recipe_sha256, source_sha256, **FEATURE_OPTIONS
        )
        labels = []
        for start in tqdm(range(0, test_data.height, 512), desc="Test inference"):
            labels.extend(
                final_pipeline.predict(test_data["text"][start : start + 512].to_list()).tolist()
            )
        predictions = test_data.select("id").with_columns(pl.Series("label", labels))
        exported = export_submission(
            predictions,
            test_data.select("id"),
            sample_submission,
            submission_path,
            {
                **input_identity,
                "model_sha256": model_record["model_sha256"],
                "candidate": asdict(selected),
                "seed": SEED,
                "model_class_order": final_pipeline.classes_.tolist(),
                "training_rows": training_data.height,
            },
        )
        display({"submission": str(submission_path.relative_to(ROOT)), "rows": exported.height})
else:
    display({"submission_status": "not executed"})

{'submission_status': 'not executed'}